# Hr Across Sessions

Heart rate compared across sessions (baseline + 3 sessions).

**Reads:** `data/individual/ (one participant, per session)`  
**Shared code:** `import mms` (loaders in `mms.io`, metrics in `mms.hrv` / `mms.stats`).

In [ ]:
import pandas as pd

DATA = '../data/individual/processed'

# load baseline HR
baseline_hr = pd.read_csv(f'{DATA}/hr.csv')

# load HR data
hr_01 = pd.read_csv(f'{DATA}/hr_01.csv')
hr_02 = pd.read_csv(f'{DATA}/hr_02.csv')
hr_03 = pd.read_csv(f'{DATA}/hr_03.csv')

# clean HR data
def clean_hr_data(hr_data):
    hr_data_clean = hr_data[hr_data['confidence'] == 1.0]
    return hr_data_clean

baseline_hr_clean = clean_hr_data(baseline_hr)
hr_01_clean = clean_hr_data(hr_01)
hr_02_clean = clean_hr_data(hr_02)
hr_03_clean = clean_hr_data(hr_03)

# average per session
baseline_avg_hr = baseline_hr_clean['heart_rate'].mean()
avg_hr_01 = hr_01_clean['heart_rate'].mean()
avg_hr_02 = hr_02_clean['heart_rate'].mean()
avg_hr_03 = hr_03_clean['heart_rate'].mean()

# baseline difference
hr_difference_01 = avg_hr_01 - baseline_avg_hr
hr_difference_02 = avg_hr_02 - baseline_avg_hr
hr_difference_03 = avg_hr_03 - baseline_avg_hr

# tachycardia threshold (AHA)
hr_anxiety_threshold = 100

# check HR anxiety
anxiety_hr_baseline = baseline_avg_hr > hr_anxiety_threshold
anxiety_hr_01 = avg_hr_01 > hr_anxiety_threshold
anxiety_hr_02 = avg_hr_02 > hr_anxiety_threshold
anxiety_hr_03 = avg_hr_03 > hr_anxiety_threshold

# display results
print(f'Baseline Average HR: {baseline_avg_hr:.2f} BPM - {"Anxiety" if anxiety_hr_baseline else "Normal"}')
print(f'Session 1 Average HR: {avg_hr_01:.2f} BPM - {"Anxiety" if anxiety_hr_01 else "Normal"}')
print(f'Difference from Baseline in Session 1: {hr_difference_01:.2f} BPM')
print(f'Session 2 Average HR: {avg_hr_02:.2f} BPM - {"Anxiety" if anxiety_hr_02 else "Normal"}')
print(f'Difference from Baseline in Session 2: {hr_difference_02:.2f} BPM')
print(f'Session 3 Average HR: {avg_hr_03:.2f} BPM - {"Anxiety" if anxiety_hr_03 else "Normal"}')
print(f'Difference from Baseline in Session 3: {hr_difference_03:.2f} BPM')

In [ ]:
import pandas as pd
import numpy as np

DATA = '../data/individual/processed'

# physiological bounds
IBI_MIN = 300
IBI_MAX = 2000

def clean_ibi_data(ibi_data):
    return ibi_data[(ibi_data['ibi'] > IBI_MIN) & (ibi_data['ibi'] < IBI_MAX)]

baseline_ibi = clean_ibi_data(pd.read_csv(f'{DATA}/ibi.csv'))
ibi_01 = clean_ibi_data(pd.read_csv(f'{DATA}/ibi_01.csv'))
ibi_02 = clean_ibi_data(pd.read_csv(f'{DATA}/ibi_02.csv'))
ibi_03 = clean_ibi_data(pd.read_csv(f'{DATA}/ibi_03.csv'))

def calculate_rmssd(ibi_values):
    return np.sqrt(np.mean(np.diff(ibi_values) ** 2))

def calculate_sdnn(ibi_values):
    return np.std(ibi_values, ddof=1)

def hrv(ibi_df):
    vals = ibi_df['ibi'].dropna().values
    return calculate_rmssd(vals), calculate_sdnn(vals)

rmssd_baseline, sdnn_baseline = hrv(baseline_ibi)
rmssd_01, sdnn_01 = hrv(ibi_01)
rmssd_02, sdnn_02 = hrv(ibi_02)
rmssd_03, sdnn_03 = hrv(ibi_03)

RMSSD_THRESHOLD = 20
SDNN_THRESHOLD = 50

def anxiety_flags(rmssd, sdnn):
    return rmssd < RMSSD_THRESHOLD, sdnn < SDNN_THRESHOLD

print(f'Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms')
print(f'Session 1 RMSSD: {rmssd_01:.2f} ms, SDNN: {sdnn_01:.2f} ms')
print(f'Session 2 RMSSD: {rmssd_02:.2f} ms, SDNN: {sdnn_02:.2f} ms')
print(f'Session 3 RMSSD: {rmssd_03:.2f} ms, SDNN: {sdnn_03:.2f} ms')

for label, rmssd, sdnn in [('Baseline', rmssd_baseline, sdnn_baseline),
                            ('Session 1', rmssd_01, sdnn_01),
                            ('Session 2', rmssd_02, sdnn_02),
                            ('Session 3', rmssd_03, sdnn_03)]:
    ar, asd = anxiety_flags(rmssd, sdnn)
    print(f'{label} RMSSD Anxiety: {"Yes" if ar else "No"}, SDNN Anxiety: {"Yes" if asd else "No"}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

def load_hr(path):
    # confidence filter + convert once
    df = pd.read_csv(path)
    df = df[df['confidence'] == 1.0].copy()
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    return df

hr_01 = load_hr(f'{DATA}/hr_01.csv')
hr_02 = load_hr(f'{DATA}/hr_02.csv')
hr_03 = load_hr(f'{DATA}/hr_03.csv')

psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

for df in [psychometric_01, psychometric_02, psychometric_03]:
    for col in ['Question Start Time', 'Question Answer Time']:
        df[col] = pd.to_datetime(df[col], utc=True, errors='coerce').dt.tz_convert(None)

question_types = ['HADS', 'STAI-T', 'STAI-S', 'BFI', 'FQ']

psychometric_data = {
    qt: tuple(df[df['Type'] == qt].reset_index(drop=True)
              for df in [psychometric_01, psychometric_02, psychometric_03])
    for qt in question_types
}

def avg_hr_per_question(psy, hr):
    # interval average, not point estimate
    psy = psy.copy()
    psy['avg_hr'] = [
        hr.loc[
            (hr['datetime'] >= row['Question Start Time']) &
            (hr['datetime'] <= row['Question Answer Time']),
            'heart_rate'
        ].mean()
        for _, row in psy.iterrows()
    ]
    return psy

def process_and_plot(type_key):
    data_01, data_02, data_03 = psychometric_data[type_key]

    data_01 = avg_hr_per_question(data_01, hr_01)
    data_02 = avg_hr_per_question(data_02, hr_02)
    data_03 = avg_hr_per_question(data_03, hr_03)

    for d in [data_01, data_02, data_03]:
        d['Question Number'] = d['Test'].str.extract(r'(\d+)')[0].fillna(0).astype(int)
        d['Time Difference'] = (d['Question Start Time'] - d['Question Start Time'].min()).dt.total_seconds()

    plt.figure(figsize=(14, 7))
    for d, s in [(data_01, 1), (data_02, 2), (data_03, 3)]:
        plt.plot(d['Time Difference'], d['avg_hr'], label=f'HR - {type_key} Session {s}', marker='o')
        for i in range(len(d)):
            plt.annotate(int(d['Question Number'].iloc[i]),
                         (d['Time Difference'].iloc[i], d['avg_hr'].iloc[i]))

    plt.title(f'Heart Rate During {type_key} Questions')
    plt.xlabel('Time Difference (seconds)')
    plt.ylabel('Heart Rate')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

for q_type in question_types:
    process_and_plot(q_type)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

question_types_count = {'HADS': 14, 'STAI-S': 20, 'STAI-T': 20, 'BFI': 10, 'FQ': 24}

sessions = []
for i in range(1, 4):
    hr = pd.read_csv(f'{DATA}/hr_{i:02d}.csv')
    hr = hr[hr['confidence'] == 1.0].copy()
    hr['datetime'] = pd.to_datetime(hr['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    psy = pd.read_csv(f'{PSY}/Psychometric_Test_Results_{i:02d}.csv')
    psy['Question Start Time'] = pd.to_datetime(psy['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
    psy['Question Answer Time'] = pd.to_datetime(psy['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
    sessions.append((hr, psy))

def plot_heart_rate_for_category(hr_data, psy_data, category_name, expected_questions, session_num, window_size=10):
    category_data = psy_data[psy_data['Type'] == category_name].copy()

    segments, question_times = [], []
    for _, row in category_data.iterrows():
        mask = (hr_data['datetime'] >= row['Question Start Time']) & (hr_data['datetime'] <= row['Question Answer Time'])
        seg = hr_data.loc[mask]
        if not seg.empty:
            segments.append(seg)
        question_times.append(row['Question Answer Time'])

    if len(question_times) != expected_questions:
        print(f"Warning: {category_name} has {len(question_times)} questions, expected {expected_questions}")

    category_hr = pd.concat(segments, ignore_index=True) if segments else pd.DataFrame()
    category_hr = category_hr.sort_values('datetime').reset_index(drop=True)
    category_hr['smoothed_heart_rate'] = category_hr['heart_rate'].rolling(window=window_size).mean()

    plt.figure(figsize=(12, 6))
    plt.plot(category_hr['datetime'], category_hr['smoothed_heart_rate'], label='Heart Rate', color='b')

    for i, time in enumerate(question_times, start=1):
        nearest_idx = category_hr['datetime'].searchsorted(time)
        if nearest_idx < len(category_hr):
            hr_val = category_hr.iloc[nearest_idx]['smoothed_heart_rate']
            plt.scatter(time, hr_val, color='red')
            plt.text(time, hr_val + 1, f'Q{i}', rotation=45, ha='right')
        else:
            plt.axvline(x=time, color='red', linestyle='--', alpha=0.5)

    plt.xlabel('Time')
    plt.ylabel('Heart Rate (bpm)')
    plt.title(f'Heart Rate During {category_name} - Session {session_num:02d}')
    plt.xticks(rotation=45)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

for i, (hr, psy) in enumerate(sessions, 1):
    for category, expected in question_types_count.items():
        plot_heart_rate_for_category(hr, psy, category, expected, i)